## Team Name/Member: Bernard MULAW
#### Date: 30/10/2026

In [ ]:
import platform
print(platform.python_version())

3.13.18


In [ ]:
%pip -q install "opencv-python" " git+https://github.com/openai/CLIP.git" "segment-anything"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# https://pypi.org/project/segment-anything-py/, https://github.com/openai/CLIP
import torch
import clip
from segment_anything import SamPredictor, sam_model_registry
from PIL import Image
import os

In [ ]:
print(torch.__version__)
torch.cuda.is_available()

2.11.0+cpu


False

In [ ]:
device = "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

100%|███████████████████████████████████████| 338M/338M [00:05<00:00, 64.5MiB/s]


In [ ]:
class_names = ["cat", "dog", "bird", "horse"]
text = clip.tokenize([f"a photo of a {c}" for c in class_names]).to(device)

In [ ]:
train_dir = "/content/drive/MyDrive/Colab/computer_vision/task1_dataset/test"
correct = 0
total = 0

per_class_accuracy = []

for cls in class_names:
  cls_dir = os.path.join(train_dir, cls)
  new_class_accuracy_count = 0
  for img_name in sorted(os.listdir(cls_dir)):
    if not img_name.endswith(".jpg"):
      continue
    img_path = os.path.join(cls_dir, img_name)
    # clip's preprocessing (TODO: look up unsqueeze?)
    image = preprocess(Image.open(img_path)).unsqueeze(0).to(device)

    with torch.no_grad():
      logits, _ = model(image, text)
      probs = logits.softmax(dim=-1)[0]
    pred = class_names[probs.argmax()]
    correct += (pred == cls)
    total += 1
    new_class_accuracy_count += (pred == cls)
  per_class_accuracy += [{cls: new_class_accuracy_count/25}]

print(f"Accuracy: {correct}/{total} = {correct/total * 100:.1f}%")
print(per_class_accuracy)

Accuracy: 345/400 = 86.2%
[{'cat': 3.56}, {'dog': 3.28}, {'bird': 3.12}, {'horse': 3.84}]


## Task 2.1: Zero-Shot Classification

In [ ]:
test_dir = "/content/drive/MyDrive/Colab/computer_vision/task1_dataset/test/random10"
# TODO : Load CLIP model (ViT -B /32) --> DONE
# TODO : Define class names --> DONE
# TODO : Load all images from folder --> DONE
# TODO : For each image :
# - Preprocess image  --> DONE
# - Tokenize text --> DONE
# - Compute similarity --> DONE
# - Output : image_name -> predicted_class ( confidence ) --> DONE

correct_pred = 0
total_predicted = 0
for img_name in sorted(os.listdir(test_dir)):
  if not img_name.endswith(".jpg"):
    continue
  img_path = test_dir + "/" + img_name
  image = preprocess(Image.open(img_path)).unsqueeze(0).to(device)
  text = clip.tokenize([f"a photo of a {c}" for c in class_names]).to(device)

  with torch.no_grad():
    logits, _ = model(image, text)
    probs = logits.softmax(dim=-1)[0]

  pred = class_names[probs.argmax()]
  confidence = round(float(max(probs)), 3)
  actual = img_name.split('_')[1]
  correct_pred += (pred == actual)
  total_predicted += 1
  print("image_class =", actual, 'model pred =', pred, "with confidence of", confidence)


print(f"Accuracy: {correct_pred}/{total_predicted} = {correct_pred/total_predicted *100:.1f}%")




image_class = bird model pred = bird with confidence of 0.894
image_class = bird model pred = bird with confidence of 0.979
image_class = bird model pred = bird with confidence of 0.995
image_class = cat model pred = cat with confidence of 0.634
image_class = cat model pred = dog with confidence of 0.388
image_class = dog model pred = dog with confidence of 0.981
image_class = dog model pred = dog with confidence of 0.963
image_class = dog model pred = dog with confidence of 0.993
image_class = horse model pred = horse with confidence of 0.974
image_class = horse model pred = horse with confidence of 0.967
Accuracy: 9/10 = 90.0%
